[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amaciasagro/GIT-RemoteSensing/blob/master/soils_v2.ipynb)
# 🌱 Soil Analyzer — v2
**Autor:** Ariel Macías | Agrónomo · GIS & Remote Sensing

Este notebook permite analizar los suelos de un lote agrícola usando datos oficiales de la USDA-NRCS (Soil Data Mart) y Google Earth Engine.

### Flujo de trabajo
| Celda | Descripción |
|-------|-------------|
| 0 | Parámetros de configuración |
| 1 | Inicialización GEE y dibujo del lote |
| 2 | Descarga espacial de unidades de suelo (WFS) y mapa con textura coloreada |
| 3 | Reporte agronómico completo: textura, pH, CEC, AWC + gráficos Plotly |

### Propiedades consultadas (USDA SDA)
| Propiedad | Descripción |
|-----------|-------------|
| Arena / Limo / Arcilla | Granulometría horizonte superficial (0–20 cm) |
| Materia Orgánica (%) | Contenido de MO superficial |
| pH | Reacción del suelo |
| CEC | Capacidad de intercambio catiónico (cmol/kg) |
| AWC | Agua disponible (cm/cm) |
| Clase textural | Clasificada automáticamente desde granulometría |

> **Fuente de datos:** [USDA-NRCS Soil Data Access](https://sdmdataaccess.nrcs.usda.gov/) · [UC Davis SoilWeb](https://casoilresource.lawr.ucdavis.edu/soilweb-apps/)
>
> ⚠️ Los datos USDA-NRCS tienen cobertura en **Estados Unidos**. Para lotes en Argentina u otros países, estos servicios no devuelven datos.

In [ ]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# Editá estas variables antes de correr el notebook.
# Para mayor seguridad podés cargar el proyecto desde entorno:
#   import os; GEE_PROJECT = os.environ.get('GEE_PROJECT', 'tu-proyecto')
# ============================================================

GEE_PROJECT  = 'your_project'   # <-- Reemplazá con tu project ID de GEE
CENTRO_LAT   = 33.584              # Latitud del centro del mapa inicial
CENTRO_LON   = -101.845            # Longitud del centro del mapa inicial
ZOOM_INICIAL = 14

print("✅ Configuración cargada.")
print(f"   Proyecto GEE : {GEE_PROJECT}")
print(f"   Centro mapa  : ({CENTRO_LAT}, {CENTRO_LON})")

---
## Paso 1 · Dibujar el lote
Usá la herramienta de polígono (panel izquierdo) para delimitar el lote y luego ejecutá la Celda 2.

In [ ]:
# ============================================================
# CELDA 1 — INICIALIZACIÓN GEE Y DIBUJO DEL LOTE
# ============================================================
import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
from shapely.geometry import Polygon
from IPython.display import display, HTML
from contextlib import redirect_stdout

# Suprimir warnings de SSL (requests a USDA usan verify=False)
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# --- Autenticación GEE (una sola vez) ---
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")
except Exception:
    print("🔑 Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")

print("\n--- PASO 1: DIBUJÁ TU LOTE ---")
print("Usá el polígono de la izquierda para dibujar. Cuando termines, ejecutá la Celda 2.")

Draw_Map = geemap.Map(center=[CENTRO_LAT, CENTRO_LON], zoom=ZOOM_INICIAL)
Draw_Map.add_basemap('SATELLITE')
display(Draw_Map)

---
## Paso 2 · Descarga espacial y mapa de unidades de suelo
Se conecta al servicio WFS de la USDA-NRCS y descarga los polígonos de unidades cartográficas dentro del lote.
Cada unidad se colorea según su **clase textural** dominante.

In [ ]:
# ============================================================
# CELDA 2 — DESCARGA WFS Y MAPA COLOREADO POR TEXTURA
# ============================================================

# 1. Verificación del dibujo
if Draw_Map.user_roi is None:
    raise ValueError("⚠️ No se detectó ningún dibujo. Volvé a la Celda 1, dibujá el lote y ejecutá esta celda.")

# 2. Geometría del AOI
roi_coords = Draw_Map.user_roi.getInfo()['coordinates'][0]
lote_geom  = ee.Geometry.Polygon([roi_coords])
centroid   = lote_geom.centroid(maxError=1).coordinates().getInfo()
centro_lote = [centroid[1], centroid[0]]  # [lat, lon]

shapely_poly = Polygon(roi_coords)
minx, miny, maxx, maxy = shapely_poly.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"

# 3. Consulta WFS a USDA Soil Data Mart
wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params  = {
    'SERVICE'     : 'WFS',
    'VERSION'     : '1.0.0',
    'REQUEST'     : 'GetFeature',
    'TYPENAME'    : 'MapunitPoly',
    'BBOX'        : bbox_str,
    'SRSNAME'     : 'EPSG:4326',
    'OUTPUTFORMAT': 'GML3'
}

print("📡 Conectando al Soil Data Mart USDA (WFS)...")
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError(f"❌ Error al conectar con USDA: {response.text}")

gdf_soils = gpd.read_file(io.BytesIO(response.content))

if gdf_soils.empty:
    raise ValueError("⚠️ No se encontraron datos de suelo en esta área. Recordá que USDA-NRCS solo tiene cobertura en EE.UU.")

gdf_lote    = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[shapely_poly])
gdf_clipped = gpd.overlay(gdf_soils, gdf_lote, how='intersection')
print(f"✅ {len(gdf_clipped)} unidades de suelo descargadas y recortadas al lote.")

# 4. Consulta tabular rápida para obtener textura dominante (para colorear el mapa)
mukeys_list = gdf_clipped['mukey'].unique().tolist()
mukeys_str  = str(tuple(mukeys_list)).replace(',)', ')')
if len(mukeys_list) == 1:
    mukeys_str = f"({mukeys_list[0]})"

sql_textura = f"""
SELECT c.mukey, c.compname, c.comppct_r,
       ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str}
  AND c.majcompflag = 'Yes'
  AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

def clasificar_textura(sand, silt, clay):
    """Clasificación USDA de textura a partir de % arena, limo y arcilla."""
    if any(v is None for v in [sand, silt, clay]):
        return 'Sin datos'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40:
        return 'Arcilloso'
    if c >= 40 and si >= 40:
        return 'Arcillo limoso'
    if c >= 35 and s >= 45:
        return 'Arcillo arenoso'
    if c >= 27 and s <= 45 and si >= 28:
        return 'Franco arcilloso'
    if c >= 27 and s >= 45:
        return 'Franco arcillo arenoso'
    if c >= 27 and si >= 60:
        return 'Franco arcillo limoso'
    if si >= 50 and c < 27:
        return 'Franco limoso'
    if si >= 80 and c < 12:
        return 'Limoso'
    if s >= 85 and c < 10:
        return 'Arenoso'
    if s >= 70 and c < 15:
        return 'Franco arenoso'
    return 'Franco'

# Paleta de colores por clase textural
PALETA_TEXTURA = {
    'Arcilloso'            : '#8B0000',
    'Arcillo limoso'       : '#B22222',
    'Arcillo arenoso'      : '#CD5C5C',
    'Franco arcilloso'     : '#D2691E',
    'Franco arcillo arenoso': '#A0522D',
    'Franco arcillo limoso': '#C68642',
    'Franco limoso'        : '#DEB887',
    'Limoso'               : '#F5DEB3',
    'Franco'               : '#8FBC8F',
    'Franco arenoso'       : '#F4A460',
    'Arenoso'              : '#EDC9Af',
    'Sin datos'            : '#AAAAAA',
}

df_tex = pd.DataFrame()
try:
    r_tex = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_textura, 'format': 'JSON'},
        verify=False, timeout=30
    )
    if r_tex.status_code == 200 and 'Table' in r_tex.json():
        df_tex = pd.DataFrame(
            r_tex.json()['Table'],
            columns=['mukey','compname','comppct_r','sand','silt','clay']
        )
        df_tex['textura'] = df_tex.apply(
            lambda row: clasificar_textura(row['sand'], row['silt'], row['clay']), axis=1
        )
except Exception as e:
    print(f"⚠️ No se pudo obtener textura para colorear el mapa: {e}")

# 5. Mapa coloreado por clase textural
mapa_suelos = geemap.Map(center=centro_lote, zoom=14)
mapa_suelos.add_basemap('SATELLITE')

# Agregar cada unidad con su color
gdf_map = gdf_clipped[['mukey', 'musym', 'geometry']].copy()

if not df_tex.empty:
    # Asignar textura dominante por mukey
    tex_dom = df_tex.sort_values('comppct_r', ascending=False).drop_duplicates('mukey')[['mukey','textura']]
    
    # ✅ FIX: homogeneizar tipo de 'mukey' en ambos DataFrames antes del merge
    gdf_map['mukey'] = gdf_map['mukey'].astype(str)
    tex_dom = tex_dom.copy()
    tex_dom['mukey'] = tex_dom['mukey'].astype(str)
    
    gdf_map = gdf_map.merge(tex_dom, on='mukey', how='left')
    gdf_map['textura'] = gdf_map['textura'].fillna('Sin datos')
else:
    gdf_map['textura'] = 'Sin datos'

# Agregar una capa por clase textural (para leyenda legible)
for textura, grupo in gdf_map.groupby('textura'):
    color = PALETA_TEXTURA.get(textura, '#AAAAAA')
    style = {'color': color, 'weight': 1, 'fillColor': color, 'fillOpacity': 0.55}
    hover = {'fillOpacity': 0.85}
    with redirect_stdout(io.StringIO()):
        mapa_suelos.add_gdf(
            grupo, layer_name=f"Textura: {textura}",
            style=style, hover_style=hover
        )

# Límite del lote en blanco
with redirect_stdout(io.StringIO()):
    mapa_suelos.add_gdf(
        gdf_lote, layer_name='Límite del lote',
        style={'color': 'white', 'weight': 2, 'fillOpacity': 0}
    )

print("✅ Mapa generado. Las capas se pueden alternar desde el panel de capas.")
display(mapa_suelos)

---
## Paso 3 · Reporte agronómico completo
Consulta el servicio tabular SDA para obtener textura, MO, pH, CEC y AWC por serie de suelo.
Genera un reporte HTML expandible con gráficos de composición granulométrica por unidad.

In [ ]:
# ============================================================
# CELDA 3 — REPORTE AGRONÓMICO COMPLETO
# ============================================================

print("\n--- GENERANDO REPORTE AGRONÓMICO ---")

# 1. Calcular área por unidad cartográfica
gdf_temp = gdf_clipped.copy()
gdf_temp['hectareas'] = gdf_temp.to_crs(epsg=3857).area / 10000
resumen_mukey = (
    gdf_temp.groupby(['mukey', 'musym'])
            .agg({'hectareas': 'sum'})
            .reset_index()
            .sort_values('hectareas', ascending=False)
)
total_ha = resumen_mukey['hectareas'].sum()

mukeys_list = resumen_mukey['mukey'].unique().tolist()
mukeys_str  = str(tuple(mukeys_list)).replace(',)', ')')
if len(mukeys_list) == 1:
    mukeys_str = f"({mukeys_list[0]})"

# 2. Consulta SQL extendida: granulometría + MO + pH + CEC + AWC
sql_completo = f"""
SELECT
    c.mukey,
    c.compname,
    c.comppct_r,
    ch.sandtotal_r,
    ch.silttotal_r,
    ch.claytotal_r,
    ch.om_r,
    ch.ph1to1h2o_r,
    ch.cec7_r,
    ch.awc_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str}
  AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

series_data = pd.DataFrame()
try:
    print("📡 Consultando propiedades de suelo (SDA)...")
    r = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_completo, 'format': 'JSON'},
        verify=False, timeout=30
    )
    if r.status_code == 200 and 'Table' in r.json():
        series_data = pd.DataFrame(
            r.json()['Table'],
            columns=['mukey','serie','pct','arena','limo','arcilla','mo','ph','cec','awc']
        )
        # Convertir columnas numéricas
        for col in ['pct','arena','limo','arcilla','mo','ph','cec','awc']:
            series_data[col] = pd.to_numeric(series_data[col], errors='coerce')
        # Agregar clase textural
        series_data['textura'] = series_data.apply(
            lambda row: clasificar_textura(row['arena'], row['limo'], row['arcilla']), axis=1
        )
        print(f"✅ Datos obtenidos para {series_data['serie'].nunique()} series de suelo.")
    else:
        print(f"⚠️ El servidor SDA devolvió código: {r.status_code}")
except Exception as e:
    print(f"❌ Error al consultar SDA: {e}")

# 3. Función auxiliar para colorear valores de pH
def color_ph(val):
    if val is None or pd.isna(val): return '#333'
    if val < 5.5:   return '#e74c3c'  # muy ácido
    if val < 6.5:   return '#e67e22'  # ácido
    if val <= 7.5:  return '#27ae60'  # neutro (óptimo)
    if val <= 8.5:  return '#f39c12'  # alcalino
    return '#c0392b'                  # muy alcalino

def fmt(val, decimales=1):
    """Formatea un valor numérico o devuelve '—' si es nulo."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return '—'
    return f"{float(val):.{decimales}f}"

# 4. Construcción del reporte HTML
html = """
<div style='font-family: sans-serif; max-width: 1000px;'>
<h2 style='color:#2c3e50; border-bottom:3px solid #2c3e50; padding-bottom:8px;'>
    📊 Reporte Agronómico de Suelos
</h2>
<p style='color:#555; font-size:13px;'>
    Fuente: USDA-NRCS Soil Data Access · Horizonte superficial (0–20 cm aprox.)
</p>
<table style='width:100%; border-collapse:collapse; margin-bottom:20px; font-size:13px;'>
<thead>
    <tr style='background:#2c3e50; color:white;'>
        <th style='padding:8px; text-align:left;'>Unidad</th>
        <th style='padding:8px;'>Superficie</th>
        <th style='padding:8px;'>% Lote</th>
        <th style='padding:8px;'>Textura dominante</th>
    </tr>
</thead>
<tbody>
"""

for _, fila in resumen_mukey.iterrows():
    pct_lote = (fila['hectareas'] / total_ha) * 100
    detalles = series_data[series_data['mukey'].astype(str) == str(fila['mukey'])] if not series_data.empty else pd.DataFrame()
    # Textura de la serie dominante
    tex_dom = detalles.sort_values('pct', ascending=False).iloc[0]['textura'] if not detalles.empty else 'Sin datos'
    color_tex = PALETA_TEXTURA.get(tex_dom, '#aaa')
    html += f"""
    <tr style='border-bottom:1px solid #eee;'>
        <td style='padding:8px; font-weight:bold;'>{fila['musym']}</td>
        <td style='padding:8px; text-align:center;'>{fila['hectareas']:.2f} ha</td>
        <td style='padding:8px; text-align:center;'>{pct_lote:.1f}%</td>
        <td style='padding:8px; text-align:center;'>
            <span style='background:{color_tex}; color:white; padding:3px 10px; border-radius:12px; font-size:12px;'>
                {tex_dom}
            </span>
        </td>
    </tr>
    """

html += "</tbody></table>"

# 5. Secciones expandibles por unidad con tabla detallada
for _, fila in resumen_mukey.iterrows():
    pct_lote = (fila['hectareas'] / total_ha) * 100
    detalles = series_data[series_data['mukey'].astype(str) == str(fila['mukey'])] if not series_data.empty else pd.DataFrame()

    html += f"""
    <details style='margin-bottom:12px; border:1px solid #bdc3c7; border-radius:6px; overflow:hidden;'>
        <summary style='background:#2c3e50; padding:14px 18px; font-weight:bold;
                        cursor:pointer; color:#fff; font-size:14px;'>
            📍 Unidad {fila['musym']} &nbsp;·&nbsp; {fila['hectareas']:.2f} ha ({pct_lote:.1f}%)
        </summary>
        <div style='padding:16px; background:#fff;'>
        <table style='width:100%; border-collapse:collapse; font-size:13px;'>
            <thead>
                <tr style='background:#ecf0f1; color:#2c3e50;'>
                    <th style='padding:9px; text-align:left; border:1px solid #ddd;'>Serie</th>
                    <th style='border:1px solid #ddd; padding:9px;'>%</th>
                    <th style='border:1px solid #ddd; padding:9px;'>Arena%</th>
                    <th style='border:1px solid #ddd; padding:9px;'>Limo%</th>
                    <th style='border:1px solid #ddd; padding:9px;'>Arcilla%</th>
                    <th style='border:1px solid #ddd; padding:9px;'>Textura</th>
                    <th style='border:1px solid #ddd; padding:9px; color:#d35400;'>MO%</th>
                    <th style='border:1px solid #ddd; padding:9px;'>pH</th>
                    <th style='border:1px solid #ddd; padding:9px;'>CEC</th>
                    <th style='border:1px solid #ddd; padding:9px;'>AWC</th>
                </tr>
            </thead>
            <tbody>
    """

    if detalles.empty:
        html += "<tr><td colspan='10' style='padding:10px; color:#888; text-align:center;'>Sin datos tabulares para esta unidad.</td></tr>"
    else:
        for _, s in detalles.iterrows():
            nombre_s     = str(s['serie']).strip()
            nombre_s_url = urllib.parse.quote(nombre_s.lower())
            url_ficha    = f"https://casoilresource.lawr.ucdavis.edu/sde/?series={nombre_s_url}#osd"
            ph_val       = s['ph'] if not pd.isna(s['ph']) else None
            tex_s        = s['textura']
            color_t      = PALETA_TEXTURA.get(tex_s, '#aaa')

            html += f"""
                <tr style='border-bottom:1px solid #eee;'>
                    <td style='padding:9px; font-weight:bold; border:1px solid #ddd;'>
                        <a href='{url_ficha}' target='_blank'
                           style='color:#3498db; text-decoration:none;'>{nombre_s}</a> 🔗
                    </td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['pct'],0)}%</td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['arena'])}</td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['limo'])}</td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['arcilla'])}</td>
                    <td style='text-align:center; border:1px solid #ddd;'>
                        <span style='background:{color_t}; color:white; padding:2px 8px;
                                     border-radius:10px; font-size:11px;'>{tex_s}</span>
                    </td>
                    <td style='text-align:center; border:1px solid #ddd;
                               color:#d35400; font-weight:bold;'>{fmt(s['mo'],2)}</td>
                    <td style='text-align:center; border:1px solid #ddd;
                               color:{color_ph(ph_val)}; font-weight:bold;'>{fmt(ph_val)}</td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['cec'],1)}</td>
                    <td style='text-align:center; border:1px solid #ddd;'>{fmt(s['awc'],3)}</td>
                </tr>"""

    html += "</tbody></table></div></details>"

html += "</div>"
display(HTML(html))

print("\n📊 Generando gráficos de composición granulométrica...")